In [2]:
import pandas as pd

In [3]:


# 1. Forcer l'affichage de TOUTES les lignes (pas de coupure au milieu)
pd.set_option('display.max_rows', None)

# 2. Forcer l'affichage de TOUTES les colonnes
pd.set_option('display.max_columns', None)

# 3. Forcer l'affichage de tout le texte sans le tronquer (ex: les longs noms de pôles)
pd.set_option('display.max_colwidth', None)


In [4]:
traitement = pd.read_csv("/home/onyxia/stage_2A/data/traitement_clean.csv")

In [ ]:
print("--- Répartition des années d'obtention du FEDER ---")
print(traitement['annee'].value_counts().sort_index())

--- Répartition des années d'obtention du FEDER ---
annee
2007      762
2008     3454
2009    10025
2010    11010
2011     8571
2012     6266
2013     4504
2014     2085
2015      492
Name: count, dtype: int64


In [5]:
controls = pd.read_csv("/home/onyxia/stage_2A/api_geo_checkpoint.csv")

In [6]:
merge4_filtre=pd.read_csv("/home/onyxia/stage_2A/merge4_filtre.csv")

In [7]:
controls_clean = pd.merge(
    merge4_filtre, 
    controls, 
    left_on='bvdid', 
    right_on='siren_clean', 
    how='inner'
)

In [ ]:
print("--- Années de bilans disponibles pour les TRAITÉS ---")
print(traitement['year'].value_counts().sort_index().head(20))

print("\n--- Années de bilans disponibles pour les CONTRÔLES ---")
print(controls_clean['year'].value_counts().sort_index().head(20))

--- Années de bilans disponibles pour les TRAITÉS ---
year
1986       2
1987       2
1988       2
1989       3
1990       3
1991       3
1992       9
1993       9
1994      10
1995     672
1996    1044
1997    1125
1998    1290
1999    1358
2000    1437
2001    1566
2002    1674
2003    1876
2004    2024
2005    2134
Name: count, dtype: int64

--- Années de bilans disponibles pour les CONTRÔLES ---
year
2002    309784
2003    354147
2004    387657
2005    424388
2006    467566
2007    512468
2008    560226
2009    595870
2010    632566
2011    659802
2012    679649
2013    660056
2014    547244
2015    474154
2016    369093
2017     93958
2018        87
Name: count, dtype: int64


In [ ]:
print("--- Répartition des années d'obtention du FEDER ---")
print(traitement['annee'].value_counts().sort_index())

--- Répartition des années d'obtention du FEDER ---
annee
2007      762
2008     3454
2009    10025
2010    11010
2011     8571
2012     6266
2013     4504
2014     2085
2015      492
Name: count, dtype: int64


In [30]:
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURATION DE TES DONNÉES (CORRIGÉE : AGE CALCULÉ DYNAMIQUEMENT)
# ==============================================================================
id_firm = 'bvdid'
# Les variables numériques dont on veut la moyenne au format _pre
variables_numeriques_pre = ['total_assets', 'number_of_employees', 'sales']
# Les variables à suivre dans le futur (post)
variables_post = ['total_assets', 'number_of_employees', 'sales']

# Filtrer les cohortes de traités valides (2007 à 2013)
df_treated_filtered = traitement[traitement['annee'].between(2007, 2013)].copy()

liste_cohortes = []

# Boucle sur chaque cohorte (année de subvention)
for c in range(2007, 2014):
    print(f"Restructuration sécurisée de la cohorte {c}...")
    
    # --------------------------------------------------------------------------
    # A. GROUPE DES TRAITÉS
    # --------------------------------------------------------------------------
    df_t = df_treated_filtered[df_treated_filtered['annee'] == c]
    df_t_pre = df_t[df_t['year'].isin([c-2, c-1])]
    
    # Numérique (moyenne)
    t_pre_num = df_t_pre.groupby(id_firm)[variables_numeriques_pre].mean().add_suffix('_pre')
    # Catégoriel + Date de création
    # 1. Dans la partie GROUPE DES TRAITÉS (autour de la ligne 20) :
# Ajoute 'category_of_the_company' à la liste des colonnes extraites
    t_pre_cat = df_t_pre.groupby(id_firm)[['code_departement', 'bvd_major_sector', 'standardised_legal_form', 'date_of_incorporation', 'category_of_the_company']].first()

# 2. Dans la partie GROUPE DE CONTRÔLE (autour de la ligne 42) :
# Ajoute 'category_of_the_company' à la liste des colonnes extraites
    t_pre_cat = t_pre_cat.rename(columns={'code_departement': 'departement'})
    
    t_pre = t_pre_num.join(t_pre_cat)
    
    # Périodes POST
    t_post1 = df_t[df_t['year'] == c+1].groupby(id_firm)[variables_post].mean().add_suffix('_post1')
    t_post2 = df_t[df_t['year'] == c+2].groupby(id_firm)[variables_post].mean().add_suffix('_post2')
    t_post3 = df_t[df_t['year'] == c+3].groupby(id_firm)[variables_post].mean().add_suffix('_post3')
    
    t_combined = t_pre.join([t_post1, t_post2, t_post3], how='inner')
    t_combined['treated'] = 1
    t_combined['cohort'] = c
    
    # CALCUL DYNAMIQUE DE L'ÂGE AU MOMENT DE LA COHORTE
    annee_inc_t = pd.to_numeric(t_combined['date_of_incorporation'], errors='coerce') // 100
    t_combined['age_pre'] = (c - 1) - annee_inc_t
    
    # --------------------------------------------------------------------------
    # B. GROUPE DE CONTRÔLE
    # --------------------------------------------------------------------------
    df_c_pre = controls_clean[controls_clean['year'].isin([c-2, c-1])]
    
    # Numérique (moyenne)
    c_pre_num = df_c_pre.groupby(id_firm)[variables_numeriques_pre].mean().add_suffix('_pre')
    # Catégoriel + Date de création
    c_pre_cat = df_c_pre.groupby(id_firm)[['departement', 'bvd_major_sector', 'standardised_legal_form', 'date_of_incorporation', 'category_of_the_company']].first()
      # Harmonisation géo
    
    c_pre = c_pre_num.join(c_pre_cat)
    
    # Périodes POST
    c_post1 = controls_clean[controls_clean['year'] == c+1].groupby(id_firm)[variables_post].mean().add_suffix('_post1')
    c_post2 = controls_clean[controls_clean['year'] == c+2].groupby(id_firm)[variables_post].mean().add_suffix('_post2')
    c_post3 = controls_clean[controls_clean['year'] == c+3].groupby(id_firm)[variables_post].mean().add_suffix('_post3')
    
    c_combined = c_pre.join([c_post1, c_post2, c_post3], how='inner')
    c_combined['treated'] = 0
    c_combined['cohort'] = c
    
    # CALCUL DYNAMIQUE DE L'ÂGE AU MOMENT DE LA COHORTE
    annee_inc_c = pd.to_numeric(c_combined['date_of_incorporation'], errors='coerce') // 100
    c_combined['age_pre'] = (c - 1) - annee_inc_c
    
    # --------------------------------------------------------------------------
    # C. EMPILAGE
    # --------------------------------------------------------------------------
    cohorte_df = pd.concat([t_combined, c_combined]).reset_index()
    liste_cohortes.append(cohorte_df)

# Assemblage final
df_final_matching = pd.concat(liste_cohortes, ignore_index=True)

print("\n--- Analyse de la base RESTRUCTURÉE avec Âge Calculé ---")
print(f"Nombre total de lignes : {len(df_final_matching)}")
print(df_final_matching['treated'].value_counts())

Restructuration sécurisée de la cohorte 2007...
Restructuration sécurisée de la cohorte 2008...
Restructuration sécurisée de la cohorte 2009...
Restructuration sécurisée de la cohorte 2010...
Restructuration sécurisée de la cohorte 2011...
Restructuration sécurisée de la cohorte 2012...
Restructuration sécurisée de la cohorte 2013...

--- Analyse de la base RESTRUCTURÉE avec Âge Calculé ---
Nombre total de lignes : 1238960
treated
0    1237129
1       1831
Name: count, dtype: int64


In [31]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

print("=== SOUS-ÉTAPE 3.1 : Préparation et passage au Logarithme ===")
df_step = df_final_matching.copy()

# 1. Calcul des logs
df_step['log_assets'] = np.log1p(df_step['total_assets_pre'])
df_step['log_employees'] = np.log1p(df_step['number_of_employees_pre'])
df_step['log_sales'] = np.log1p(df_step['sales_pre'])
df_step['log_age'] = np.log1p(df_step['age_pre'])

# 2. Sécurité : On remplace les valeurs infinies (inf, -inf) générées par les logs par des NaNs
df_step = df_step.replace([np.inf, -np.inf], np.nan)

# 3. Nettoyage strict de TOUS les NaNs avant de donner la base au modèle
vars_strictes = ['log_assets', 'log_employees', 'log_sales', 'log_age', 
                 'departement', 'bvd_major_sector', 'standardised_legal_form']
df_step = df_step.dropna(subset=vars_strictes).copy()

print(f"Nombre de lignes après nettoyage strict des NaNs et infinis : {len(df_step)}")


print("\n=== SOUS-ÉTAPE 3.2 : Application du Support Commun Strict ===")
# On élimine les modalités de départements/secteurs trop rares (moins de 3 traités ou 3 contrôles)
colonnes_categories = ['departement', 'bvd_major_sector', 'standardised_legal_form', 'cohort']

for col in colonnes_categories:
    # HARMONISATION DES TYPES : On force en texte propre (on supprime le ".0" des floats si présent)
    df_step[col] = df_step[col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    
    # Calcul de la table croisée
    table_croisee = pd.crosstab(df_step[col], df_step['treated'])
    
    # SÉCURITÉ ANTI-CRASH : Si le groupe 0 ou 1 est absent de la table croisée, on le simule à 0
    if 0 not in table_croisee.columns:
        table_croisee[0] = 0
    if 1 not in table_croisee.columns:
        table_croisee[1] = 0
        
    # Sélection des catégories valides (au moins 3 contrôles ET 3 traités)
    categories_valibles = table_croisee[(table_croisee[0] >= 3) & (table_croisee[1] >= 3)].index
    df_step = df_step[df_step[col].isin(categories_valibles)].copy()

print(f"Nombre de lignes final prêtes pour le modèle : {len(df_step)}")
print("Répartition des classes :")
print(df_step['treated'].value_counts())


print("\n=== SOUS-ÉTAPE 3.3 : Estimation du Modèle Logit ===")
# Dans la Sous-étape 3.3, modifie la formule :
formule_log = (
    "treated ~ log_assets + log_employees + log_sales + log_age "
    "+ C(cohort) + C(departement) + C(bvd_major_sector) + C(standardised_legal_form) + C(category_of_the_company)"
)

# On lance le modèle sur la base parfaitement nettoyée et harmonisée
modele_logit = smf.logit(formule_log, data=df_step).fit(method='bfgs', maxiter=300)


print("\n=== SOUS-ÉTAPE 3.4 : Extraction et alignement du pscore ===")
# On utilise predict(df_step) pour forcer l'alignement parfait des index pandas
df_step['pscore'] = modele_logit.predict(df_step)

print("✅ Étape réussie ! Le pscore a été intégré sans aucun bug géo-temporel.")
print("\n=== DIAGNOSTIC QUALITÉ DU SCORE DE PROPENSION ===")
print(df_step.groupby('treated')['pscore'].describe())

# On sauvegarde dans la variable globale pour l'étape du matching
df_final_matching = df_step.copy()

=== SOUS-ÉTAPE 3.1 : Préparation et passage au Logarithme ===


/opt/python/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/python/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/python/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/python/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Nombre de lignes après nettoyage strict des NaNs et infinis : 658406

=== SOUS-ÉTAPE 3.2 : Application du Support Commun Strict ===
Nombre de lignes final prêtes pour le modèle : 537601
Répartition des classes :
treated
0    536541
1      1060
Name: count, dtype: int64

=== SOUS-ÉTAPE 3.3 : Estimation du Modèle Logit ===
Optimization terminated successfully.
         Current function value: 0.009911
         Iterations: 206
         Function evaluations: 220
         Gradient evaluations: 220

=== SOUS-ÉTAPE 3.4 : Extraction et alignement du pscore ===
✅ Étape réussie ! Le pscore a été intégré sans aucun bug géo-temporel.

=== DIAGNOSTIC QUALITÉ DU SCORE DE PROPENSION ===
            count      mean       std           min       25%       50%  \
treated                                                                   
0        536541.0  0.001769  0.008864  9.358792e-09  0.000059  0.000234   
1          1060.0  0.108719  0.214435  1.853650e-05  0.003366  0.016429   

              75% 

In [32]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import numpy as np

print("=== ÉTAPE 4.1 : Séparation STRICTE et filtrage des entreprises ===")
# Séparation
treated_df = df_final_matching[df_final_matching['treated'] == 1].drop_duplicates(subset=['bvdid', 'cohort']).copy()
control_df_raw = df_final_matching[df_final_matching['treated'] == 0].drop_duplicates(subset=['bvdid', 'cohort']).copy()

# Sécurité : pas de traité dans le bassin de contrôle
liste_bvdid_traites = treated_df['bvdid'].unique()
control_df = control_df_raw[~control_df_raw['bvdid'].isin(liste_bvdid_traites)].copy()

print(f"Entreprises traitées : {len(treated_df)}")
print(f"Bassin de contrôle initial : {len(control_df)}")

print("\n=== ÉTAPE 4.2 : Matching Exact (Cohorte + Secteur) + Proximité Pscore ===")

liste_paires = []
id_paire = 0
traites_sans_jumeaux = 0

# On groupe les traités par Cohorte et par Secteur Majeur
# Remplace la ligne des groupes par celle-ci :
# Étape 4.2 : Matching Exact (Cohorte + Secteur + Département + Catégorie)

# 1. On regroupe par 4 critères :
groupes_traites = treated_df.groupby(['cohort', 'bvd_major_sector', 'departement', 'category_of_the_company'])

# 2. On modifie la recherche des contrôles pour inclure la catégorie :
for (cohorte, secteur, dept, cat), t_sub in groupes_traites:
    c_sub = control_df[(control_df['cohort'] == cohorte) & 
                       (control_df['bvd_major_sector'] == secteur) & 
                       (control_df['departement'] == dept) &
                       (control_df['category_of_the_company'] == cat)]
    
    # ... reste du code identique ...
    # ... reste du code identique ...
    
    # Si aucun contrôle n'est disponible dans cette combinaison exacte, on passe (support commun strict)
    if len(c_sub) == 0:
        traites_sans_jumeaux += len(t_sub)
        continue
        
    # On applique le plus proche voisin sur le pscore au sein de ce sous-groupe
    # (On autorise le remplacement au cas où un contrôle est le meilleur jumeau pour deux traités)
    nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
    nn.fit(c_sub[['pscore']].values)
    
    distances, indices = nn.kneighbors(t_sub[['pscore']].values)
    
    # Assemblage des couples
    for i in range(len(t_sub)):
        ligne_traite = t_sub.iloc[[i]].copy()
        
        idx_controle_voisin = indices[i][0]
        ligne_controle = c_sub.iloc[[idx_controle_voisin]].copy()
        
        # Attribution de l'ID de paire et de la distance
        ligne_traite['pair_id'] = id_paire
        ligne_controle['pair_id'] = id_paire
        ligne_traite['match_distance'] = distances[i][0]
        ligne_controle['match_distance'] = distances[i][0]
        
        liste_paires.append(ligne_traite)
        liste_paires.append(ligne_controle)
        id_paire += 1

# Création de la base finale unifiée
df_matched_final = pd.concat(liste_paires, ignore_index=True)

# Tri visuel logique
df_matched_final = df_matched_final.sort_values(by=['pair_id', 'treated'], ascending=[True, False]).reset_index(drop=True)

# Réorganisation des colonnes clés au début
colonnes_cles = ['pair_id', 'treated', 'bvdid', 'cohort', 'bvd_major_sector', 'departement', 'pscore', 'sales_pre', 'number_of_employees_pre']
autres_cols = [c for c in df_matched_final.columns if c not in colonnes_cles]
df_matched_final = df_matched_final[colonnes_cles + autres_cols]

print(f"✅ Matching par bloc terminé.")
print(f"Nombre d'entreprises traitées éliminées faute de jumeau dans leur secteur/cohorte : {traites_sans_jumeaux}")
print(f"Taille de l'échantillon apparié final : {len(df_matched_final)} lignes ({len(df_matched_final)//2} paires).")

print("\n=== VÉRIFICATION DES 6 PREMIÈRES LIGNES (3 PAIRES) ===")
display(df_matched_final[colonnes_cles].head(6))

=== ÉTAPE 4.1 : Séparation STRICTE et filtrage des entreprises ===
Entreprises traitées : 1060
Bassin de contrôle initial : 534408

=== ÉTAPE 4.2 : Matching Exact (Cohorte + Secteur) + Proximité Pscore ===
✅ Matching par bloc terminé.
Nombre d'entreprises traitées éliminées faute de jumeau dans leur secteur/cohorte : 103
Taille de l'échantillon apparié final : 1914 lignes (957 paires).

=== VÉRIFICATION DES 6 PREMIÈRES LIGNES (3 PAIRES) ===


,pair_id,treated,bvdid,cohort,bvd_major_sector,departement,pscore,sales_pre,number_of_employees_pre
0,0,1,401918446,2007,"Chemicals, rubber, plastics, non-metallic products",12,0.008659,14464000.0,57.0
1,0,0,395047467,2007,"Chemicals, rubber, plastics, non-metallic products",12,0.003143,1699500.0,6.0
2,1,1,326700549,2007,"Chemicals, rubber, plastics, non-metallic products",81,0.000342,2589500.0,19.0
3,1,0,329060586,2007,"Chemicals, rubber, plastics, non-metallic products",81,0.000353,2732500.0,15.0
4,2,1,435266598,2007,"Chemicals, rubber, plastics, non-metallic products",87,0.035380,680000.0,4.5
5,2,0,410246607,2007,"Chemicals, rubber, plastics, non-metallic products",87,0.012086,1970892.5,37.0


In [33]:
def check_balance(df):
    # Variables à vérifier (celles qui ont servi à créer le pscore)
    variables = ['log_sales', 'log_employees', 'log_assets', 'log_age']
    
    print(f"{'Variable':<20} | {'Moyenne Traités':<15} | {'Moyenne Contrôles':<15} | {'Différence (%)':<15}")
    print("-" * 75)
    
    for var in variables:
        mean_t = df[df['treated'] == 1][var].mean()
        mean_c = df[df['treated'] == 0][var].mean()
        
        # Différence relative en pourcentage
        diff_pct = ((mean_t - mean_c) / mean_c) * 100
        
        print(f"{var:<20} | {mean_t:<15.4f} | {mean_c:<15.4f} | {diff_pct:>13.2f}%")

print("=== DIAGNOSTIC D'ÉQUILIBRE (Balance Check) ===")
check_balance(df_matched_final)

print("\n--- Interprétation :")
print("Si les différences sont proches de 0% ou faibles, ton matching est EXCELLENT.")
print("Si tu vois des différences > 10-15%, cela signifie que tes groupes restent déséquilibrés.")

=== DIAGNOSTIC D'ÉQUILIBRE (Balance Check) ===
Variable             | Moyenne Traités | Moyenne Contrôles | Différence (%) 
---------------------------------------------------------------------------
log_sales            | 14.9321         | 14.9649         |         -0.22%
log_employees        | 3.3880          | 3.3809          |          0.21%
log_assets           | 15.1956         | 15.1731         |          0.15%
log_age              | 2.6184          | 2.8995          |         -9.70%

--- Interprétation :
Si les différences sont proches de 0% ou faibles, ton matching est EXCELLENT.
Si tu vois des différences > 10-15%, cela signifie que tes groupes restent déséquilibrés.


In [ ]:
import statsmodels.formula.api as smf

# On définit la formule pour une des variables (exemple : log_sales)
# On regarde l'évolution entre la moyenne pré et post1 (ou post3 selon ton intérêt)
# La DiD simple : (Post - Pre) Treated - (Post - Pre) Control
# Mais on peut utiliser une régression avec fixed effects si tu as assez de données.

# Voici une structure de régression robuste pour tester l'effet sur le CA (sales) :
formule_did = "sales_post1 ~ treated + sales_pre + log_employees + log_age + C(cohort) + C(bvd_major_sector)"

model_did = smf.ols(formule_did, data=df_final_matched).fit(cov_type='cluster', cov_kwds={'groups': df_final_matching['pair_id']})

print(model_did.summary())